# 🥇 Gold Layer — Development Notebook

Use this notebook to test and develop the Gold layer logic locally.

**Flow:** Silver data → Technical indicators + Summary aggregations → gold tables

In [ ]:
from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("GoldLayer_Dev")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"Spark version: {spark.version}")

## 1. Load Silver Output

In [ ]:
import os
from datetime import date, timedelta
import random

if os.path.exists("data/test/silver_output"):
    silver_df = spark.read.parquet("data/test/silver_output")
    print(f"Loaded Silver output: {silver_df.count()} rows")
else:
    # Generate 60 days of sample data for meaningful indicators
    rows = []
    for symbol, base_price in [("AAPL", 185.0), ("GOOGL", 140.0)]:
        price = base_price
        for i in range(60):
            d = date(2024, 1, 2) + timedelta(days=i)
            change = random.uniform(-3, 3)
            price = max(price + change, 50)
            o = round(price - random.uniform(0, 2), 2)
            h = round(price + random.uniform(0, 3), 2)
            l = round(price - random.uniform(0, 3), 2)
            c = round(price, 2)
            v = random.randint(20000000, 60000000)
            rows.append((symbol, d, o, h, l, c, c, v, 0.0, 1.0))
    
    columns = ["symbol", "date", "open", "high", "low", "close", "adjusted_close", "volume", "dividend_amount", "split_coefficient"]
    silver_df = spark.createDataFrame(rows, columns)
    print(f"Generated sample Silver data: {silver_df.count()} rows (60 days × 2 symbols)")

silver_df.show(5, truncate=False)
print(f"Symbols: {[r.symbol for r in silver_df.select('symbol').distinct().collect()]}")

## 2. Technical Indicator Functions
Test each indicator individually — modify these to tweak your logic.

In [ ]:
def add_sma(df, column, window_size, output_col):
    """Simple Moving Average."""
    w = Window.partitionBy("symbol").orderBy("date").rowsBetween(-(window_size - 1), 0)
    return df.withColumn(output_col, F.round(F.avg(column).over(w), 4))

def add_rsi(df, column="close", period=14):
    """Relative Strength Index."""
    w = Window.partitionBy("symbol").orderBy("date")
    w_period = Window.partitionBy("symbol").orderBy("date").rowsBetween(-(period - 1), 0)
    df = df.withColumn("_prev_close", F.lag(column, 1).over(w))
    df = df.withColumn("_change", F.col(column) - F.col("_prev_close"))
    df = df.withColumn("_gain", F.when(F.col("_change") > 0, F.col("_change")).otherwise(0))
    df = df.withColumn("_loss", F.when(F.col("_change") < 0, -F.col("_change")).otherwise(0))
    df = df.withColumn("_avg_gain", F.avg("_gain").over(w_period))
    df = df.withColumn("_avg_loss", F.avg("_loss").over(w_period))
    df = df.withColumn("rsi_14",
        F.when(F.col("_avg_loss") == 0, 100.0)
         .otherwise(F.round(100.0 - (100.0 / (1.0 + F.col("_avg_gain") / F.col("_avg_loss"))), 4))
    )
    return df.drop("_prev_close", "_change", "_gain", "_loss", "_avg_gain", "_avg_loss")

def add_bollinger_bands(df, column="close", window_size=20, num_std=2):
    """Bollinger Bands."""
    w = Window.partitionBy("symbol").orderBy("date").rowsBetween(-(window_size - 1), 0)
    df = df.withColumn("_bb_sma", F.avg(column).over(w))
    df = df.withColumn("_bb_std", F.stddev(column).over(w))
    df = df.withColumn("bollinger_upper", F.round(F.col("_bb_sma") + num_std * F.col("_bb_std"), 4))
    df = df.withColumn("bollinger_lower", F.round(F.col("_bb_sma") - num_std * F.col("_bb_std"), 4))
    return df.drop("_bb_sma", "_bb_std")

def add_daily_return(df, column="close"):
    """Daily return percentage."""
    w = Window.partitionBy("symbol").orderBy("date")
    df = df.withColumn("_prev", F.lag(column, 1).over(w))
    df = df.withColumn("daily_return_pct",
        F.when(F.col("_prev").isNotNull() & (F.col("_prev") != 0),
               F.round(((F.col(column) - F.col("_prev")) / F.col("_prev")) * 100, 4))
         .otherwise(None)
    )
    return df.drop("_prev")

print("✅ Indicator functions defined")

## 3. Compute Indicators — Step by Step

In [ ]:
# SMA
df = add_sma(silver_df, "close", 20, "sma_20")
df = add_sma(df, "close", 50, "sma_50")
print("✅ SMA (20, 50) computed")
df.select("symbol", "date", "close", "sma_20", "sma_50").show(5)

In [ ]:
# EMA (approximation via SMA)
df = add_sma(df, "close", 12, "ema_12")
df = add_sma(df, "close", 26, "ema_26")

# MACD
df = df.withColumn("macd", F.round(F.col("ema_12") - F.col("ema_26"), 4))
df = add_sma(df, "macd", 9, "macd_signal")
print("✅ EMA + MACD computed")
df.select("symbol", "date", "ema_12", "ema_26", "macd", "macd_signal").show(5)

In [ ]:
# RSI
df = add_rsi(df, column="close", period=14)
print("✅ RSI (14) computed")
df.select("symbol", "date", "close", "rsi_14").show(5)

In [ ]:
# Bollinger Bands
df = add_bollinger_bands(df, column="close", window_size=20, num_std=2)
print("✅ Bollinger Bands computed")
df.select("symbol", "date", "close", "bollinger_upper", "bollinger_lower").show(5)

In [ ]:
# Daily Returns
df = add_daily_return(df, column="close")
print("✅ Daily returns computed")
df.select("symbol", "date", "close", "daily_return_pct").show(5)

## 4. Gold Table 1: Technical Indicators

In [ ]:
indicators_df = df.select(
    "symbol", "date", "close",
    "sma_20", "sma_50", "ema_12", "ema_26",
    "rsi_14", "macd", "macd_signal",
    "bollinger_upper", "bollinger_lower",
    "daily_return_pct",
)

print(f"Gold Indicators: {indicators_df.count()} rows")
indicators_df.show(10, truncate=False)

## 5. Gold Table 2: Daily Summary

In [ ]:
summary_df = (
    silver_df
    .groupBy("symbol")
    .agg(
        F.count("date").alias("total_trading_days"),
        F.min("date").alias("first_date"),
        F.max("date").alias("last_date"),
        F.round(F.avg("close"), 2).alias("avg_close"),
        F.min("low").alias("all_time_low"),
        F.max("high").alias("all_time_high"),
        F.avg("volume").cast("long").alias("avg_daily_volume"),
        F.sum("volume").alias("total_volume"),
        F.round(F.stddev("close"), 4).alias("close_stddev"),
        F.round(F.avg("adjusted_close"), 2).alias("avg_adjusted_close"),
    )
)

print(f"Gold Summary: {summary_df.count()} rows")
summary_df.show(truncate=False)

## 6. Spot Check — Validate Indicators

In [ ]:
# Check for nulls in key columns
for col_name in ["sma_20", "rsi_14", "macd", "bollinger_upper"]:
    null_count = indicators_df.filter(F.col(col_name).isNull()).count()
    total = indicators_df.count()
    pct = round((null_count / total) * 100, 1)
    print(f"{col_name}: {null_count}/{total} nulls ({pct}%) — expected for first N rows")

# Check RSI is between 0 and 100
bad_rsi = indicators_df.filter(
    (F.col("rsi_14").isNotNull()) &
    ((F.col("rsi_14") < 0) | (F.col("rsi_14") > 100))
).count()
print(f"\nRSI out of range (0-100): {bad_rsi} rows {'✅' if bad_rsi == 0 else '❌'}")

In [ ]:
# spark.stop()